In [0]:
# ===== PARAMETERS =====
runs_table = "test_automation_runs2"
results_table = "test_automation_results2"
filter_states = []
# ======================

from pyspark.sql.functions import col, desc, row_number
from pyspark.sql.window import Window
from collections import defaultdict
import pandas as pd

chain_order = [
    "paymentPending", "appealSubmitted",
    "awaitingRespondentEvidence(a)", "awaitingRespondentEvidence(b)",
    "caseUnderReview", "reasonsForAppealSubmitted", "listing",
    "prepareForHearing", "decision", "decided(a)",
    "ftpaSubmitted(a)", "decided(b)", "ftpaSubmitted(b)", "ftpaDecided",
    "remitted", "ended"
]

active_states = filter_states if filter_states else chain_order

runs_df = spark.table(runs_table)
results_df = spark.table(results_table)

trans_runs = runs_df.filter(col("run_by_automation_name") == "Transformation_Tests")
all_runs_list = trans_runs.orderBy(desc("run_start_datetime")).collect()
run_lookup = {r.run_id: r for r in all_runs_list}
all_run_ids = [r.run_id for r in all_runs_list]

if not all_run_ids:
    print(f"No runs found in {runs_table}")
else:
    all_results = results_df.filter(col("run_id").isin(all_run_ids)).toPandas()
    run_to_state = {r.run_id: r.state_under_test for r in all_runs_list}
    all_results["run_state"] = all_results["run_id"].map(run_to_state)
    all_results["status_upper"] = all_results["status"].str.upper().str.strip()

    # Reclassify on the fly
    NO_DATA_PATTERNS = ["NO RECORDS TO TEST", "NO MATCHING TEST DATA", "DOES NOT EXIST IN THE",
                        "NO RECORDS FOUND", "NO DATA AVAILABLE", "NO DATA TO TEST",
                        "NO DATA EXISTS FOR", "UNRESOLVED_COLUMN", "FAILED TO SETUP DATA"]
    ERROR_PATTERNS = ["IS NOT DEFINED", "TEST CRASHED:"]

    def reclassify(row):
        status = row["status_upper"]
        if status == "PASS": return "PASS"
        if status in ("NO_DATA", "ERROR"): return status
        msg = str(row.get("message", "") or "").upper()
        for p in NO_DATA_PATTERNS:
            if p in msg: return "NO_DATA"
        for p in ERROR_PATTERNS:
            if p in msg: return "ERROR"
        return status

    all_results["status_upper"] = all_results.apply(reclassify, axis=1)

    if all_results.empty:
        print("No results found")
    else:
        # ============================================================
        # Build field x state matrix — best result per field+state across ALL runs
        # ============================================================
        field_state_best = defaultdict(dict)
        # Track every result per field+state+run for drill-down
        field_state_runs = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

        for _, row in all_results.iterrows():
            field = str(row.get("test_field", "") or "").strip()
            from_state = str(row.get("test_from_state", "") or "").strip()
            rs = row["status_upper"]
            run_id = row.get("run_id", "")
            if not field or not from_state: continue
            if from_state not in active_states: continue

            field_state_runs[field][from_state][run_id].append({
                "status": rs,
                "test_name": row.get("test_name", ""),
                "message": str(row.get("message", ""))[:80],
            })

            priority = {"PASS": 4, "FAIL": 3, "ERROR": 2, "NO_DATA": 1, "": 0}
            current = field_state_best[field].get(from_state, "")
            if priority.get(rs, 0) > priority.get(current, 0):
                field_state_best[field][from_state] = rs

        states_in_data = sorted(
            set(s for fd in field_state_best.values() for s in fd.keys()),
            key=lambda s: chain_order.index(s) if s in chain_order else 999
        )
        all_fields = sorted(field_state_best.keys())

        # ============================================================
        # Field verdicts
        # ============================================================
        field_verdicts = {}
        for field in all_fields:
            statuses = list(field_state_best[field].values())
            all_pass = all(s == "PASS" for s in statuses)
            has_fail = "FAIL" in statuses
            has_error = "ERROR" in statuses

            if all_pass: verdict = "PASSED"
            elif has_fail or has_error: verdict = "FAILED"
            else: verdict = "NEEDS DATA"

            field_verdicts[field] = {
                "verdict": verdict,
                "states": len(statuses),
                "pass": statuses.count("PASS"),
                "fail": statuses.count("FAIL"),
                "error": statuses.count("ERROR"),
                "nodata": statuses.count("NO_DATA"),
            }

        count_passed = sum(1 for v in field_verdicts.values() if v["verdict"] == "PASSED")
        count_failed = sum(1 for v in field_verdicts.values() if v["verdict"] == "FAILED")
        count_needs_data = sum(1 for v in field_verdicts.values() if v["verdict"] == "NEEDS DATA")
        count_total_fields = len(field_verdicts)

        result_pass = (all_results["status_upper"] == "PASS").sum()
        result_fail = (all_results["status_upper"] == "FAIL").sum()
        result_error = (all_results["status_upper"] == "ERROR").sum()
        result_nodata = (all_results["status_upper"] == "NO_DATA").sum()
        result_total = len(all_results)

        # ============================================================
        # Build field rows with run-level drill-down
        # ============================================================
        def verdict_sort(field):
            order = {"FAILED": 0, "NEEDS DATA": 1, "PASSED": 2}
            return (order.get(field_verdicts[field]["verdict"], 3), field)

        field_rows = ""
        row_idx = 0
        for field in sorted(all_fields, key=verdict_sort):
            v = field_verdicts[field]
            verdict = v["verdict"]
            v_css = {"PASSED": "v-pass", "FAILED": "v-fail", "NEEDS DATA": "v-nodata"}[verdict]

            state_cells = ""
            for state in states_in_data:
                val = field_state_best[field].get(state, "")
                if val == "PASS": state_cells += '<td class="hm-pass">&#10003;</td>'
                elif val == "FAIL": state_cells += '<td class="hm-fail">&#10007;</td>'
                elif val == "ERROR": state_cells += '<td class="hm-err">!</td>'
                elif val == "NO_DATA": state_cells += '<td class="hm-nd">&mdash;</td>'
                else: state_cells += '<td class="hm-empty"></td>'

            field_rows += f'''<tr class="field-row" onclick="toggleDetail({row_idx})" style="cursor:pointer" title="Click to expand">
                <td><b>{field}</b></td>
                <td class="{v_css}">{verdict}</td>
                {state_cells}
                <td>{v["states"]}</td>
                <td class="pass">{v["pass"]}</td>
                <td class="fail">{v["fail"]}</td>
                <td class="error">{v["error"]}</td>
                <td class="nodata">{v["nodata"]}</td>
            </tr>\n'''

            # Drill-down: group by state, then by run
            detail_html = ""
            for state in states_in_data:
                runs_for_state = field_state_runs[field].get(state, {})
                if not runs_for_state: continue

                for rid, results_list in sorted(runs_for_state.items(), key=lambda x: str(run_lookup.get(x[0], {}))):
                    run_info = run_lookup.get(rid)
                    run_date = str(run_info.run_start_datetime)[:16] if run_info else "?"
                    run_state_label = run_to_state.get(rid, "?")
                    p = sum(1 for r in results_list if r["status"] == "PASS")
                    f = sum(1 for r in results_list if r["status"] == "FAIL")
                    e = sum(1 for r in results_list if r["status"] == "ERROR")
                    nd = sum(1 for r in results_list if r["status"] == "NO_DATA")
                    total = len(results_list)

                    # Run summary row
                    detail_html += f'''<tr style="background:#e8f4fd">
                        <td colspan="2"><b>{state}</b></td>
                        <td>Run: {rid[:8]}...</td>
                        <td>{run_date}</td>
                        <td><span class="pass">{p}P</span> <span class="fail">{f}F</span> <span class="error">{e}E</span> <span class="nodata">{nd}ND</span> = {total}</td>
                    </tr>\n'''

                    # Individual results
                    for r in results_list:
                        d_css = {"PASS": "pass", "FAIL": "fail", "ERROR": "error", "NO_DATA": "nodata"}.get(r["status"], "")
                        detail_html += f'''<tr>
                            <td></td>
                            <td>{r["test_name"]}</td>
                            <td class="{d_css}">{r["status"]}</td>
                            <td colspan="2">{r["message"]}</td>
                        </tr>\n'''

            if detail_html:
                field_rows += f'''<tr class="detail-row" id="detail-{row_idx}" style="display:none">
                    <td colspan="{7 + len(states_in_data)}">
                        <table style="width:100%;margin:4px 0;font-size:11px;background:#fafafa">
                            <thead><tr><th>State</th><th>Test</th><th>Run / Status</th><th>Date</th><th>Detail</th></tr></thead>
                            <tbody>{detail_html}</tbody>
                        </table>
                    </td>
                </tr>\n'''

            row_idx += 1

        state_header = "".join(
            f'<th style="writing-mode:vertical-lr;min-width:22px;padding:3px;font-size:10px">{s}</th>'
            for s in states_in_data
        )

        html = f"""
        <style>
            .csr {{font-family:Arial,sans-serif;padding:10px}}
            .csr h2 {{color:#333;border-bottom:2px solid #333;padding-bottom:5px;margin-top:25px}}
            .csr table {{border-collapse:collapse;width:100%;margin-bottom:12px;font-size:12px}}
            .csr th {{background:#2c3e50;color:white;padding:4px 6px;text-align:left;position:sticky;top:0}}
            .csr td {{border:1px solid #ddd;padding:2px 6px}}
            .csr tr:nth-child(even) {{background:#f9f9f9}}
            .csr .pass {{color:#27ae60;font-weight:bold}}
            .csr .fail {{color:#e74c3c;font-weight:bold}}
            .csr .nodata {{color:#f39c12;font-weight:bold}}
            .csr .error {{color:#7f8c8d;font-weight:bold}}
            .csr .hm-pass {{background:#d5f5e3;color:#27ae60;text-align:center;font-weight:bold}}
            .csr .hm-fail {{background:#fadbd8;color:#e74c3c;text-align:center;font-weight:bold}}
            .csr .hm-err {{background:#eee;color:#7f8c8d;text-align:center;font-weight:bold}}
            .csr .hm-nd {{background:#fef9e7;color:#f39c12;text-align:center}}
            .csr .hm-empty {{background:#f8f9fa}}
            .csr .v-pass {{background:#d5f5e3;color:#155724;font-weight:bold;text-align:center}}
            .csr .v-fail {{background:#fadbd8;color:#721c24;font-weight:bold;text-align:center}}
            .csr .v-nodata {{background:#fef9e7;color:#856404;font-weight:bold;text-align:center}}
            .csr .box {{display:inline-block;padding:12px 20px;margin:4px;border-radius:5px;color:white;font-size:16px;text-align:center}}
            .csr .box-pass {{background:#27ae60}}
            .csr .box-fail {{background:#e74c3c}}
            .csr .box-nd {{background:#f39c12}}
            .csr .box-err {{background:#7f8c8d}}
            .csr .box-total {{background:#2c3e50}}
            .csr .tp {{background:#ecf0f1;padding:12px;border-left:4px solid #2c3e50;margin:12px 0;font-size:14px}}
            .csr .field-row:hover {{background:#e8f4fd !important}}
            .csr .detail-row td {{padding:0}}
            .csr .detail-row table td {{border:1px solid #eee;padding:2px 6px}}
        </style>
        <div class="csr">
            <h2>Cross-State Coverage Report</h2>
            <p><code>{runs_table}</code> / <code>{results_table}</code> | {len(all_runs_list)} runs | {len(all_fields)} fields | {len(states_in_data)} states</p>

            <div>
                <div class="box box-pass">PASS<br><b>{result_pass}</b></div>
                <div class="box box-fail">FAIL<br><b>{result_fail}</b></div>
                <div class="box box-err">ERROR<br><b>{result_error}</b></div>
                <div class="box box-nd">NO DATA<br><b>{result_nodata}</b></div>
                <div class="box box-total">TOTAL<br><b>{result_total}</b></div>
            </div>

            <div>
                <div class="box box-pass">PASSED<br>all states<br><b>{count_passed}</b> fields</div>
                <div class="box box-fail">NOT PASSED<br>fail/error/nodata<br><b>{count_failed + count_needs_data}</b> fields</div>
                <div class="box box-total">TOTAL<br>fields<br><b>{count_total_fields}</b></div>
            </div>

            <div class="tp">
                A field <b>PASSES</b> only if it passes in every state where tested.
                Any FAIL, ERROR, or NO_DATA in any state = not passed.
                Best result per field+state across all {len(all_runs_list)} runs.
            </div>

            <h2>1. Field Coverage Matrix (click row to drill down)</h2>
            <div style="overflow-x:auto;max-height:700px;overflow-y:auto">
                <table>
                    <thead><tr>
                        <th>Field</th><th>Verdict</th>
                        {state_header}
                        <th>States</th><th>P</th><th>F</th><th>E</th><th>ND</th>
                    </tr></thead>
                    <tbody>{field_rows}</tbody>
                </table>
            </div>

            <br>
            <button onclick="downloadReport()" style="background:#2c3e50;color:white;padding:10px 25px;border:none;border-radius:5px;cursor:pointer;font-size:14px">
                Download Offline Report
            </button>
        </div>
        <script>
            function toggleDetail(idx) {{
                var row = document.getElementById('detail-' + idx);
                if (row) {{
                    row.style.display = row.style.display === 'none' ? 'table-row' : 'none';
                }}
            }}
            function downloadReport() {{
                var content = document.querySelector('.csr').outerHTML;
                var styles = document.querySelectorAll('style');
                var styleText = '';
                styles.forEach(function(s) {{ styleText += s.outerHTML; }});
                var scripts = '<script>function toggleDetail(idx){{var r=document.getElementById(\"detail-\"+idx);if(r){{r.style.display=r.style.display===\"none\"?\"table-row\":\"none\"}}}}<\\/script>';
                var full = '<!DOCTYPE html><html><head><meta charset=\"utf-8\"><title>Cross-State Coverage Report</title>' + styleText + '</head><body>' + content + scripts + '</body></html>';
                var blob = new Blob([full], {{type: 'text/html'}});
                var a = document.createElement('a');
                a.href = URL.createObjectURL(blob);
                a.download = 'cross_state_report.html';
                a.click();
            }}
        </script>
        """

        displayHTML(html)
